Our package provides data access in a Python programming environment.

Here, we will start a Clustering analysis for the Pancreatic ductal adenocarcinoma (pdac).

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# from gpnotebook.tools.standard_imports import *
import os, re,sys
import yaml
import pandas as pd
import numpy as np


In [2]:
# project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/PDAC_P_PDC000271"
project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/CCRCC_P_PDC000128"
data_dir = os.path.join(project_dir,"matrix")
meta_dir = os.path.join(project_dir,"meta")
job_dir = os.path.join(project_dir,"precomputed","cluster")
if not os.path.exists(job_dir):
    os.mkdir(job_dir)

In [3]:
data_path = os.path.join(data_dir, "DIG_nglycoform-peptide_matrix-abundances-MD_norm.tsv")
data_df = pd.read_csv(data_path,sep="\t", index_col = [0,1,2,3])
data_df

Intensity.Reference  \
Site                                               Gene   Sequence                      Glycan                            
ENSP00000431932@33;ENSP00000385235@64              LARGE2 AAALDGDPGAGPGDHNRSDCGPQPPPPPK N6H3F1S0G0            15.558626   
ENSP00000349437@2122                               IGF2R  AACAVKPQEVQMVNGTITNPINGK      N4H5F1S1G0            15.320329   
                                                                                        N4H5F1S2G0            16.431956   
                                                                                        N4H6F2S1G0            13.656542   
                                                                                        N5H6F3S1G0            14.841677   
...                                                                                                                 ...   
ENSP00000299798@314                                SLC9A5 YVEANISHK                     N5H5F1S3G0            14.147254   
ENSP00000426404@159;ENSP00000302289@209;ENSP000... EMB    YVINGTYANETK                  N6H5F1S0G0            13.913320   
ENSP00000386043@333                                LTBP1  YVQDQVAAPFQLSNHTGR            N7H4F4S1G0            12.805975   
ENSP00000261590@458                                DSG2   YVQNGTYTVK                    N5H6F1S2G0            12.929367   
ENSP00000422185@320;ENSP00000329124@148            SORCS2 YVTCAIHNCSEK                  N4H5F1S1G0            12.313897   

                                                                                                    C3L-01287_N_01  \
Site                                               Gene   Sequence                      Glycan                       
ENSP00000431932@33;ENSP00000385235@64              LARGE2 AAALDGDPGAGPGDHNRSDCGPQPPPPPK N6H3F1S0G0       14.877778   
ENSP00000349437@2122                               IGF2R  AACAVKPQEVQMVNGTITNPINGK      N4H5F1S1G0       14.785880   
                                                                                        N4H5F1S2G0       16.136839   
                                                                                        N4H6F2S1G0       13.057453   
                                                                                        N5H6F3S1G0       14.039353   
...                                                                                                            ...   
ENSP00000299798@314                                SLC9A5 YVEANISHK                     N5H5F1S3G0             NaN   
ENSP00000426404@159;ENSP00000302289@209;ENSP000... EMB    YVINGTYANETK                  N6H5F1S0G0             NaN   
ENSP00000386043@333                                LTBP1  YVQDQVAAPFQLSNHTGR            N7H4F4S1G0             NaN   
ENSP00000261590@458                                DSG2   YVQNGTYTVK                    N5H6F1S2G0             NaN   
ENSP00000422185@320;ENSP00000329124@148            SORCS2 YVTCAIHNCSEK                  N4H5F1S1G0             NaN   

                                                                                                    C3L-00561_N_01  \
Site                                               Gene   Sequence                      Glycan                       
ENSP00000431932@33;ENSP00000385235@64              LARGE2 AAALDGDPGAGPGDHNRSDCGPQPPPPPK N6H3F1S0G0       15.227525   
ENSP00000349437@2122                               IGF2R  AACAVKPQEVQMVNGTITNPINGK      N4H5F1S1G0       15.195179   
                                                                                        N4H5F1S2G0       16.113973   
                                                                                        N4H6F2S1G0       13.509876   
                                                                                        N5H6F3S1G0       14.559130   
...                                                                                                            ...   
ENSP00000299798@314  

In [4]:
meta_path= os.path.join(meta_dir, "CCRCC_meta.txt")
meta_df = pd.read_csv(meta_path,sep="\t",header=[0,1])
meta_df

,case_id,Age,Sex,Tumor_Size_cm,Histologic_Grade,Tumor_necrosis,Path_Stage_pT,Path_Stage_pN,Stage,BMI,Tobacco_smoking_history,KDM5C_mutation,VHL_mutation,BAP1_mutation,PBRM1_mutation,SETD2_mutation
,data_type,CON,BIN,CON,ORD,BIN,ORD,ORD,ORD,CON,ORD,BIN,BIN,BIN,BIN,BIN
0,C3L-00908,60,Female,10.5,G3 Poorly differentiated,Not identified,pT2,NaN,Stage II,27.85,past smoker,0,1,1,1,0
1,C3L-00004,72,Male,12.0,G3 Poorly differentiated,Present,pT3,NaN,Stage III,22.80,past smoker,0,1,0,1,1
2,C3L-00010,30,Male,6.5,G3 Poorly differentiated,Not identified,pT1,pN0,Stage I,34.15,current smoker,1,1,0,0,0
3,C3L-00011,63,Female,12.0,G4 Undifferentiated,Present,pT3,NaN,Stage IV,27.47,non-smoker,1,1,1,0,0
4,C3L-00026,65,Female,2.0,G3 Poorly differentiated,Not identified,pT1,NaN,Stage I,28.23,non-smoker,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,C3N-01646,69,Male,10.0,G3 Poorly differentiated,Not identified,pT3,NaN,Stage III,25.00,current smoker,0,0,1,0,0
99,C3N-01648,69,Male,7.5,G2 Moderately differentiated,Not identified,pT2,NaN,Stage II,28.00,past smoker,0,0,0,0,0
100,C3N-01649,51,Male,8.0,G2 Moderately differentiated,Not identified,pT3,NaN,Stage III,35.00,past smoker,1,1,0,1,0


In [5]:
meta_cols = ['case_id','Sex','Stage']
meta2 = meta_df.loc[:,meta_cols]
meta2.columns = ['Sample.ID'] + meta_cols[1:]
meta2

,Sample.ID,Sex,Stage
0,C3L-00908,Female,Stage II
1,C3L-00004,Male,Stage III
2,C3L-00010,Male,Stage I
3,C3L-00011,Female,Stage IV
4,C3L-00026,Female,Stage I
...,...,...,...
98,C3N-01646,Male,Stage III
99,C3N-01648,Male,Stage II
100,C3N-01649,Male,Stage III
101,C3N-01651,Male,Stage II


In [6]:
meta2.head(17)

,Sample.ID,Sex,Stage
0,C3L-00908,Female,Stage II
1,C3L-00004,Male,Stage III
2,C3L-00010,Male,Stage I
3,C3L-00011,Female,Stage IV
4,C3L-00026,Female,Stage I
5,C3L-00079,Male,Stage III
6,C3L-00088,Male,Stage III
7,C3L-00096,Male,Stage IV
8,C3L-00097,Male,Stage I
9,C3L-00103,Male,Stage III


In [7]:
head_cols = ['Site', 'Gene', 'Sequence', 'Glycan', 'Intensity.Reference']
samples = [i for i in data_df.columns.values if i not in head_cols]
samples = [i for i in samples if i.split('_')[0] in list(meta2['Sample.ID']) and i.split('_')[1] == 'T']
len(samples)

103

In [8]:
rows = []
for sample in samples:
    key = sample.split('_')[0]
    row = meta2[meta2['Sample.ID']==key].iloc[0]
    row['Sample.ID'] = sample
    rows.append(row)
meta3 = pd.DataFrame(rows)

In [9]:
meta3.head(17)

,Sample.ID,Sex,Stage
17,C3L-00561_T_01,Male,Stage III
42,C3L-01287_T_01,Male,Stage IV
50,C3L-01603_T_01,Male,Stage I
97,C3N-01524_T_01,Male,Stage II
92,C3N-01214_T_02,Male,Stage II
84,C3N-00834_T_02,Male,Stage I
94,C3N-01261_T_02,Male,Stage I
38,C3L-00917_T_02,Male,Stage I
53,C3L-01861_T_03,Male,Stage I
46,C3L-01352_T_03,Female,Stage I


In [10]:
meta3 = meta3.replace(np.nan,'NA')

In [11]:
top_ann_data_path = os.path.join(job_dir,'top_ann_data.tsv')
meta3.to_csv(top_ann_data_path, sep="\t", index=False)

Top annotation settings.

In [12]:

top_ann_settings = {
    'Sex': {
        'Male': 'blue',
        'Female': 'red',
        'NA': 'grey',
    },
    'Stage': {
        'Stage I': 'blue',
        'Stage II': 'green',
        'Stage III': 'orange',
        'Stage IV': 'red',
        'NA': 'grey'
    },

}
top_ann_settings_path = os.path.join(job_dir,'top_ann_settings.yml')
with open(top_ann_settings_path,'w') as f:
    yaml.dump(top_ann_settings,f,default_flow_style=False)

In [13]:
data_df.head(2)

,,,,Intensity.Reference,C3L-01287_N_01,C3L-00561_N_01,C3L-00561_T_01,C3L-01287_T_01,C3L-01603_T_01,C3N-01524_T_01,C3N-01524_N_01,C3L-01603_N_01,C3L-00359_T_01,...,C3N-00380_T_23,C3N-00310_N_23,C3N-00390_N_23,C3N-00312_T_23,C3N-00494_T_23,C3N-00390_T_23,C3N-00312_N_23,C3N-00494_N_23,C3N-00310_T_23,pool_23
Site,Gene,Sequence,Glycan,,,,,,,,,,,,,,,,,,,,,
ENSP00000431932@33;ENSP00000385235@64,LARGE2,AAALDGDPGAGPGDHNRSDCGPQPPPPPK,N6H3F1S0G0,15.558626,14.877778,15.227525,15.563940,15.135967,15.633243,15.868060,15.350302,15.362149,16.149583,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENSP00000349437@2122,IGF2R,AACAVKPQEVQMVNGTITNPINGK,N4H5F1S1G0,15.320329,14.785880,15.195179,16.608193,15.596653,15.900825,14.752611,15.019048,15.097013,16.919549,...,14.595424,13.140993,15.174944,15.047426,13.434202,14.439377,13.509151,13.918898,16.581582,15.320329


In [14]:
data_df.shape

(113794, 231)

In [15]:
samples = meta3['Sample.ID'].to_list()

In [16]:
df2 = data_df.loc[:,samples].dropna()

In [17]:
df2.shape

(1645, 103)

In [18]:
from scipy.stats import variation
rows = []
for index,row in df2.iterrows():
    rows.append([variation([np.power(2,i) for i in list(row)])])
cv_df = pd.DataFrame(rows,columns=['cv'],index= df2.index)

glycopeptides = cv_df[cv_df['cv']>0.25].index

data2 = df2[df2.index.isin(glycopeptides)]
glycopeptides =  [f'{site}@{gene}@{seq}@{glycan}' for site,gene,seq,glycan in glycopeptides]
data2.index = glycopeptides
tumor_expression_path = os.path.join(job_dir,'expression_data.tsv')
data2.to_csv(tumor_expression_path,sep='\t',index=True)

In [19]:
data2.shape

(1492, 103)

Extract tumor samples from glycopeptide expression data based on pathological status,

calculates the coefficient of variation (CV) for each glycopeptide, selects glycopeptides with CV greater than 0.25.

Map glcopeptides with cv>0.25 in tumor patients with glycan type.

In [20]:
import re,os, sys

def decide_glycan_type(g):
    m = re.finditer("([A-Z])([\d]+)", g)
    y = [(i.group(1), int(i.group(2))) for i in m]
    d = dict(y)
    glycan_type = "Other"
    if d["N"] == 2 and d["H"] >= 5 and d["F"] == 0 and d["S"] == 0 and d["G"] == 0:
        glycan_type = "HM"
    elif d["N"] >= 2 and d["H"] >= 3 and d["F"] > 0 and d["S"] == 0:
        glycan_type = "only_F"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] == 0:
        glycan_type = "only_S"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] > 0:
        glycan_type = "F+S"
    return glycan_type


In [21]:
# left annotation
# from gpnotebook.tools.glycan import decide_glycan_type

glycan_type_map = dict(zip(glycopeptides,[decide_glycan_type(i) for i in glycopeptides]))
  
left_ann_data_path =  os.path.join(job_dir,'left_annotation_data.tsv')
rows = []
for i in glycan_type_map:
    rows.append([i,glycan_type_map[i]])
left_ann_data = pd.DataFrame(rows,columns=['Glycopeptide','GlycanType'])
left_ann_data.to_csv(left_ann_data_path,sep="\t",index=False)

In [22]:
left_ann_data

,Glycopeptide,GlycanType
0,ENSP00000356671@215@SERPINC1@AAINKWVSNKTEGR@N4...,only_S
1,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N4H...,F+S
2,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N5H...,F+S
3,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,only_S
4,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,only_S
...,...,...
1487,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
1488,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
1489,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
1490,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S


Map glycan types with colors.

In [23]:

# left annotation settings, including color, order
left_ann_settings_path = os.path.join(job_dir,'left_annotation_settings.yml')
left_ann_settings = {
    "glycan_type_index" :{
    "HM": 1,
    "only_F":2,
    "only_S":3,
    "F+S":4,
    "Other":5
    },
    "glycan_type_color" : {
        "HM": 'green',
    "only_F": 'red',
    "only_S": 'purple',
    "F+S": 'orange',
    "Other": 'grey'
}
}
with open(left_ann_settings_path,'w') as f:
    yaml.dump(left_ann_settings,f,default_flow_style=False)
    

Parameters for NMF clustering.

In [24]:
nmf_parameters_path = os.path.join(job_dir, 'nmf_parameters.yml')
nmf_parameters = {
    'k_range': {
        'min': 3,
        'max': 5,
    },
    'test':{
        'nruns': 50
    },
    'opt_k':{
        'nruns': 500,
        'predefined': 0,
        'value': 4,
        'feature_prob': 0.8
    }
}
with open(nmf_parameters_path,'w') as f:
    yaml.dump(nmf_parameters,f,default_flow_style=False)

Generate a YAML configuration file (nmf_configs.yml) containing paths to various data required for NMF clustering.

In [25]:
config_data = {
    'input': {
        'expression_data': tumor_expression_path,
        'left_annotation_data': left_ann_data_path ,
        'left_annotation_settings': left_ann_settings_path,
        'top_annotation_data': top_ann_data_path,
        'top_annotatin_settings': top_ann_settings_path,
        'nmf_parameters': nmf_parameters_path
    },
    'output':{
        'out_dir': job_dir
    }
}
nmf_configs_path = os.path.join(job_dir,'nmf_configs.yml')
with open(nmf_configs_path,'w') as f:
    yaml.dump(config_data,f,default_flow_style=False)